<a href="https://colab.research.google.com/github/KarunyaVarma/Karunya-Varma/blob/main/DNN_GTPP_Regression1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# -------------------------------------------------
# 1. Reproducibility & Data Loading
# -------------------------------------------------
tf.keras.utils.set_random_seed(42)

data = pd.read_csv('/content/GTPP-DATASET.csv')
X = data[['AT', 'AP', 'DF', 'CH', 'GP', 'CPR', 'CPD', 'TTXM']].values
y = data['EP'].values

# 80:20 Holdout Split
X_train_full, X_test_holdout, y_train_full, y_test_holdout = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Fit MinMaxScaler strictly on training data
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test_holdout)

# -------------------------------------------------
# 2. Final 2-Layer MLP Model Training
# -------------------------------------------------
tf.keras.utils.set_random_seed(42)

final_model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(8,)),
    tf.keras.layers.Dense(224, activation='tanh'),
    tf.keras.layers.Dense(48, activation='tanh'),
    tf.keras.layers.Dense(1)
])

final_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001603),
    loss='mse',
    metrics=['mae']
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='loss',
    factor=0.5,
    patience=12,
    min_lr=1e-6,
    verbose=0
)

final_model.fit(
    X_train_scaled,
    y_train_full,
    epochs=200,
    batch_size=16,
    callbacks=[lr_scheduler],
    verbose=0
)

# -------------------------------------------------
# 3. Holdout Predictions & Metrics
# -------------------------------------------------
y_pred = final_model.predict(X_test_scaled, verbose=0).flatten()
residuals = y_test_holdout - y_pred

mae = mean_absolute_error(y_test_holdout, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_holdout, y_pred))
r2 = r2_score(y_test_holdout, y_pred)

print(f"Holdout Test MAE  : {mae:.4f} MW")
print(f"Holdout Test RMSE : {rmse:.4f} MW")
print(f"Holdout Test R2   : {r2:.6f}")

# -------------------------------------------------
# 4. Generate 3-Panel Publication Plots
# -------------------------------------------------
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11
})

fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), dpi=300)

# --- Panel A: Actual vs. Predicted EP ---
ax1 = axes[0]
ax1.scatter(y_test_holdout, y_pred, alpha=0.45, s=18, color='#1f77b4', edgecolors='none', label='Test Samples (n=1,579)')
min_val = min(y_test_holdout.min(), y_pred.min())
max_val = max(y_test_holdout.max(), y_pred.max())
ax1.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Ideal Parity (y = x)')

metrics_text = (
    f"$R^2$ = {r2:.4f}\n"
    f"RMSE = {rmse:.3f} MW\n"
    f"MAE = {mae:.3f} MW"
)
ax1.text(
    0.05, 0.93, metrics_text,
    transform=ax1.transAxes,
    fontsize=10.5,
    verticalalignment='top',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='#cccccc', alpha=0.92)
)
ax1.set_title('(a) Actual vs. Predicted Power Output')
ax1.set_xlabel('Actual Electrical Power, EP (MW)')
ax1.set_ylabel('Predicted Electrical Power, EP (MW)')
ax1.legend(loc='lower right', frameon=True)
ax1.grid(True, linestyle=':', alpha=0.5)

# --- Panel B: Residuals vs. Predicted Values ---
ax2 = axes[1]
ax2.scatter(y_pred, residuals, alpha=0.45, s=18, color='#2ca02c', edgecolors='none')
ax2.axhline(0, color='red', linestyle='--', lw=2, label='Zero Error Line')
ax2.axhline(2 * rmse, color='gray', linestyle=':', lw=1.2, label=f'±2 RMSE (±{2*rmse:.2f} MW)')
ax2.axhline(-2 * rmse, color='gray', linestyle=':', lw=1.2)
ax2.set_title('(b) Residual Errors vs. Predicted EP')
ax2.set_xlabel('Predicted Electrical Power, EP (MW)')
ax2.set_ylabel('Residual Error (Actual - Predicted) [MW]')
ax2.legend(loc='upper right', frameon=True, fontsize=9.5)
ax2.grid(True, linestyle=':', alpha=0.5)

# --- Panel C: Residual Distribution Histogram ---
ax3 = axes[2]
sns.histplot(residuals, bins=35, kde=True, color='#9467bd', alpha=0.55, ax=ax3)
ax3.axvline(0, color='red', linestyle='--', lw=2, label='Zero Error')
ax3.axvline(np.mean(residuals), color='black', linestyle='-.', lw=1.5, label=f'Mean = {np.mean(residuals):.3f} MW')
ax3.set_title('(c) Residual Error Distribution')
ax3.set_xlabel('Residual Error (MW)')
ax3.set_ylabel('Frequency')
ax3.legend(loc='upper right', frameon=True, fontsize=9.5)
ax3.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig('mlp_gtpp_evaluation_plots.png', dpi=300, bbox_inches='tight')
plt.show()